# 📈 Bloque 2 — Sesión 1: Distribuciones de probabilidad y forma

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD2 — Verificación estadística de la calidad de los datos  
**Duración estimada:** ~3 horas

---

### Contenidos de esta sesión
- **2.1** Distribuciones de probabilidad: normal, uniforme y binomial
- **2.2** Forma de la distribución: sesgo (skewness) y curtosis (kurtosis)

### Criterios de evaluación asociados
> **(CE a)** Se han descrito las técnicas estadísticas de análisis de la calidad técnica de los datos.  
> **(CE b)** Se han aplicado técnicas de evaluación de la calidad de los datos basadas en cálculos estadísticos para detectar la cobertura y el sesgo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm, uniform, binom

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

titanic = sns.load_dataset('titanic')
diamonds = sns.load_dataset('diamonds')
penguins = sns.load_dataset('penguins').dropna()

print('✅ Librerías y datasets cargados')

---
## 2.1 Distribuciones de probabilidad

Una **distribución de probabilidad** describe cómo se reparten los valores posibles de una variable y con qué probabilidad ocurre cada uno. Conocer la distribución de un atributo nos ayuda a:

- Elegir el modelo estadístico adecuado
- Decidir si necesitamos transformar la variable antes de entrenarla
- Detectar anomalías (valores que no encajan con la distribución esperada)
- Entender el sesgo del conjunto de datos

En la práctica, las distribuciones más relevantes para el preprocesamiento de datos en ML son tres: **normal**, **uniforme** y **binomial**.

### Distribución normal (gaussiana)

La distribución normal es la más importante en estadística. Se caracteriza por:

- Forma de campana simétrica alrededor de la media (μ)
- El 68% de los datos cae dentro de ±1σ, el 95% dentro de ±2σ, el 99.7% dentro de ±3σ (regla empírica)
- Completamente descrita por solo dos parámetros: **μ** (media) y **σ** (desviación típica)

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$

> 💡 Muchos algoritmos de ML asumen normalidad (regresión lineal, LDA, tests paramétricos). En la práctica, los datos reales rara vez son perfectamente normales, pero a veces podemos acercarnos con transformaciones.

In [ ]:
# Distribución normal: distintas medias y desviaciones típicas
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(-10, 10, 500)

# Efecto de la media (μ)
for mu, color in zip([-3, 0, 3], ['steelblue', 'crimson', 'mediumseagreen']):
    axes[0].plot(x, norm.pdf(x, loc=mu, scale=1.5), color=color, linewidth=2,
                 label=f'μ={mu}, σ=1.5')
axes[0].set_title('Efecto de la media (μ)')
axes[0].legend()
axes[0].set_ylabel('Densidad de probabilidad')

# Efecto de la desviación típica (σ)
for sigma, color in zip([0.8, 1.5, 3.0], ['steelblue', 'crimson', 'mediumseagreen']):
    axes[1].plot(x, norm.pdf(x, loc=0, scale=sigma), color=color, linewidth=2,
                 label=f'μ=0, σ={sigma}')
axes[1].set_title('Efecto de la desviación típica (σ)')
axes[1].legend()

plt.suptitle('Distribución normal — variación de parámetros', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Regla empírica 68-95-99.7
mu, sigma = 0, 1
x = np.linspace(-4, 4, 500)
y = norm.pdf(x, mu, sigma)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, y, 'navy', linewidth=2)

rangos = [(1, 'steelblue', '±1σ  68.3%'),
          (2, 'cornflowerblue', '±2σ  95.4%'),
          (3, 'lightblue', '±3σ  99.7%')]

for k, color, label in reversed(rangos):
    mask = (x >= mu - k*sigma) & (x <= mu + k*sigma)
    ax.fill_between(x, y, where=mask, alpha=0.5, color=color, label=label)

ax.set_xlabel('Desviaciones típicas desde la media')
ax.set_ylabel('Densidad')
ax.set_title('Regla empírica 68-95-99.7 de la distribución normal', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

for k in [1, 2, 3]:
    p = norm.cdf(mu + k*sigma) - norm.cdf(mu - k*sigma)
    print(f'  ±{k}σ cubre el {p*100:.1f}% de los datos')

### Distribución uniforme

En una distribución uniforme, **todos los valores en un intervalo [a, b] son igualmente probables**. La densidad de probabilidad es constante:

$$f(x) = \frac{1}{b - a} \quad \text{para } a \leq x \leq b$$

**¿Cuándo aparece en datos reales?**
- Variables generadas aleatoriamente sin sesgo (números de ID, sorteos)
- Algunas variables de ruido o transformadas (si se aplica la transformación de probabilidad integral)
- Variables discretas balanceadas (dados, monedas perfectas)

> ⚠️ Si una variable numérica real tiene distribución aproximadamente uniforme, es una señal de que puede ser un **código o identificador** disfrazado de numérico, no una magnitud real.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución uniforme teórica
x = np.linspace(-0.5, 5.5, 500)
for a, b, color in [(0, 5, 'steelblue'), (1, 3, 'crimson'), (0, 1, 'mediumseagreen')]:
    axes[0].plot(x, uniform.pdf(x, loc=a, scale=b-a), linewidth=2.5,
                 label=f'[{a}, {b}]')
axes[0].set_title('Distribución uniforme (PDF teórica)')
axes[0].set_ylabel('Densidad')
axes[0].legend()
axes[0].set_ylim(0, 1.3)

# Muestra simulada
muestra_uniforme = np.random.uniform(0, 10, size=2000)
axes[1].hist(muestra_uniforme, bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Muestra de distribución uniforme U(0,10) — n=2000')
axes[1].set_ylabel('Frecuencia')

plt.suptitle('Distribución uniforme', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Distribución binomial

La distribución binomial modela el **número de éxitos** en `n` ensayos independientes, cada uno con probabilidad de éxito `p`. Es una distribución discreta:

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

**Parámetros:** n (número de ensayos), p (probabilidad de éxito por ensayo)  
**Aplicaciones en ML:**
- Variable objetivo binaria (fraude/no fraude, spam/no spam)
- Análisis de tasas de conversión
- Simulación de muestras con desbalanceo controlado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Efecto del parámetro p (probabilidad)
n = 20
x_vals = np.arange(0, n+1)
for p, color in zip([0.2, 0.5, 0.8], ['steelblue', 'crimson', 'mediumseagreen']):
    pmf = binom.pmf(x_vals, n, p)
    axes[0].plot(x_vals, pmf, 'o-', color=color, linewidth=1.5, markersize=5,
                 label=f'p={p}')
axes[0].set_title(f'Binomial(n={n}) — efecto de p')
axes[0].set_xlabel('Número de éxitos (k)')
axes[0].set_ylabel('Probabilidad P(X=k)')
axes[0].legend()

# Relación con la normal (teorema del límite central)
n_grande = 100
p = 0.4
x_vals2 = np.arange(0, n_grande+1)
pmf2 = binom.pmf(x_vals2, n_grande, p)
axes[1].bar(x_vals2, pmf2, color='steelblue', alpha=0.7, width=1, label='Binomial(100, 0.4)')

# Aprox. normal
mu_b = n_grande * p
sigma_b = np.sqrt(n_grande * p * (1-p))
x_cont = np.linspace(20, 60, 300)
axes[1].plot(x_cont, norm.pdf(x_cont, mu_b, sigma_b), 'crimson', linewidth=2.5,
             label=f'Normal(μ={mu_b}, σ={sigma_b:.1f})')
axes[1].set_title('Convergencia a la normal (n grande)')
axes[1].set_xlabel('Número de éxitos')
axes[1].legend()

plt.suptitle('Distribución binomial', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Binomial(n=100, p=0.4) → μ = n·p = {mu_b:.0f}  |  σ = √(n·p·(1-p)) = {sigma_b:.2f}")

---
## 2.2 Forma de la distribución: sesgo y curtosis

Los estadísticos de centralidad y dispersión no capturan la **forma** de la distribución. Para eso usamos el sesgo y la curtosis.

### Sesgo (Skewness)

El sesgo mide la **asimetría** de la distribución respecto a la media. Se calcula como el tercer momento estandarizado:

$$\text{Sesgo} = \frac{1}{n} \sum_{i=1}^{n} \left(\frac{x_i - \bar{x}}{s}\right)^3$$

| Valor | Interpretación | Ejemplo típico |
|-------|---------------|---------------|
| ≈ 0 | Distribución simétrica | Altura de personas |
| > 0 | Sesgo positivo (cola derecha) | Ingresos, precios de vivienda |
| < 0 | Sesgo negativo (cola izquierda) | Calificaciones (muchos aprobados altos) |

> 💡 **Regla práctica:** si |sesgo| > 1, la asimetría es importante y puede perjudicar a modelos que asumen normalidad. Considera aplicar una transformación.

In [ ]:
# Visualización del sesgo
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Sesgo negativo
data_neg = np.concatenate([np.random.normal(8, 0.5, 800), np.random.uniform(0, 8, 200)])
# Simétrico
data_sym = np.random.normal(5, 1.5, 1000)
# Sesgo positivo (log-normal)
data_pos = np.random.lognormal(0, 1, 1000)

for ax, data, title, color in zip(
    axes,
    [data_neg, data_sym, data_pos],
    ['Sesgo negativo', 'Simétrico (sesgo ≈ 0)', 'Sesgo positivo (cola derecha)'],
    ['coral', 'steelblue', 'mediumseagreen']
):
    skew = stats.skew(data)
    ax.hist(data, bins=35, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(data), color='crimson', linestyle='--', linewidth=2, label='Media')
    ax.axvline(np.median(data), color='navy', linestyle='-', linewidth=2, label='Mediana')
    ax.set_title(f'{title}\nSesgo = {skew:.2f}')
    ax.legend(fontsize=8)

plt.suptitle('Sesgo de la distribución', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Sesgo en variables reales del Titanic y Diamonds
print("Sesgo de variables numéricas (Titanic):")
for col in ['age', 'fare', 'sibsp', 'parch']:
    s = titanic[col].dropna().skew()
    alerta = ''
    if abs(s) > 1: alerta = ' ⚠️ asimetría importante'
    elif abs(s) > 0.5: alerta = ' 〰️ asimetría moderada'
    print(f"  {col:<10} sesgo = {s:+.3f}{alerta}")

print()
print("Sesgo de variables numéricas (Diamonds):")
for col in ['price', 'carat', 'depth', 'table']:
    s = diamonds[col].skew()
    alerta = ''
    if abs(s) > 1: alerta = ' ⚠️ asimetría importante'
    elif abs(s) > 0.5: alerta = ' 〰️ asimetría moderada'
    print(f"  {col:<10} sesgo = {s:+.3f}{alerta}")

### Curtosis (Kurtosis)

La curtosis mide el **peso de las colas** de la distribución y lo "puntiaguda" que es la campana. Se basa en el cuarto momento estandarizado.

En la práctica se usa la **curtosis en exceso** (Fisher), que resta 3 para que la normal tenga valor 0:

$$\text{Curtosis exceso} = \frac{1}{n} \sum_{i=1}^{n} \left(\frac{x_i - \bar{x}}{s}\right)^4 - 3$$

| Valor | Tipo | Cola | Ejemplo |
|-------|------|------|---------|
| ≈ 0 | Mesocúrtica | Normal | Distribución normal |
| > 0 | Leptocúrtica | Colas pesadas | Precios financieros, distribución t de Student |
| < 0 | Platicúrtica | Colas ligeras | Distribución uniforme |

> ⚠️ Curtosis alta (colas pesadas) significa más outliers de los esperados en una normal. En datos financieros o de fraude es muy común.

In [ ]:
# Visualización de la curtosis
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

np.random.seed(42)
# Platicúrtica (colas ligeras)
data_plat = np.random.uniform(-3, 3, 2000)
# Mesocúrtica (normal)
data_meso = np.random.normal(0, 1, 2000)
# Leptocúrtica (colas pesadas, t con pocos grados de libertad)
data_lept = stats.t.rvs(df=3, size=2000)
# Recorte para visualización
data_lept_clip = np.clip(data_lept, -6, 6)

for ax, data, title, color in zip(
    axes,
    [data_plat, data_meso, data_lept_clip],
    ['Platicúrtica (colas ligeras)', 'Mesocúrtica (normal, K≈0)', 'Leptocúrtica (colas pesadas)'],
    ['mediumseagreen', 'steelblue', 'coral']
):
    kurt = stats.kurtosis(data)  # curtosis en exceso (Fisher)
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85, density=True)
    xr = np.linspace(data.min(), data.max(), 300)
    ax.plot(xr, norm.pdf(xr, 0, 1), 'navy', linewidth=2, linestyle='--', alpha=0.7, label='Normal ref.')
    ax.set_title(f'{title}\nCurtosis (exceso) = {kurt:.2f}')
    ax.set_xlim(-6, 6)
    ax.legend(fontsize=8)

plt.suptitle('Curtosis de la distribución', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Sesgo y curtosis juntos: función de diagnóstico de forma
def diagnostico_forma(df, columnas):
    """Calcula sesgo, curtosis y emite un diagnóstico de la forma de cada variable."""
    resultados = []
    for col in columnas:
        serie = df[col].dropna()
        skew = serie.skew()
        kurt = serie.kurtosis()  # exceso
        
        # Diagnóstico sesgo
        if abs(skew) < 0.5:
            diag_skew = 'Simétrica'
        elif abs(skew) < 1:
            diag_skew = 'Asimetría moderada'
        else:
            diag_skew = '⚠️ Asimetría importante'
        
        # Diagnóstico curtosis
        if abs(kurt) < 0.5:
            diag_kurt = 'Mesocúrtica (normal)'
        elif kurt > 0:
            diag_kurt = '⚠️ Leptocúrtica (colas pesadas)'
        else:
            diag_kurt = 'Platicúrtica (colas ligeras)'
        
        resultados.append({
            'Variable': col,
            'Sesgo': round(skew, 3),
            'Curtosis exceso': round(kurt, 3),
            'Forma sesgo': diag_skew,
            'Forma curtosis': diag_kurt
        })
    return pd.DataFrame(resultados).set_index('Variable')

print("── Titanic ──")
print(diagnostico_forma(titanic, ['age', 'fare', 'sibsp', 'parch']))
print()
print("── Diamonds ──")
print(diagnostico_forma(diamonds, ['price', 'carat', 'depth', 'table']))

---
## 🔧 Ejercicios

### Ejercicio 1 — Reconocer distribuciones

Genera tres muestras con `numpy` y clasifica cada una como normal, uniforme o binomial:
- 1.000 valores de `np.random.exponential(scale=2)`
- 1.000 valores de `np.random.normal(loc=10, scale=3)`
- 1.000 valores de `np.random.binomial(n=30, p=0.6)`

Visualiza sus histogramas. Para cada una, razona: ¿qué tipo de variable real podría seguir esta distribución?

In [ ]:
# Genera las tres muestras y traza los histogramas
# fig, axes = plt.subplots(1, 3, figsize=(14, 4))
# ...

# Pista: usa axes[i].hist() y añade el título con el tipo de distribución identificado

### Ejercicio 2 — Sesgo y curtosis en Penguins

Usa la función `diagnostico_forma()` sobre las variables numéricas del dataset `penguins`. Identifica:
- ¿Qué variables tienen asimetría importante?
- ¿Hay alguna con colas pesadas (leptocúrtica)?
- ¿Qué implicaciones tendría para un modelo de clasificación de especies?

In [ ]:
# cols_penguins = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
# diagnostico_forma(penguins, cols_penguins)

# Complementa con visualización: histogramas con seaborn
# fig, axes = plt.subplots(2, 2, figsize=(12, 8))
# for ax, col in zip(axes.flat, cols_penguins):
#     sns.histplot(penguins, x=col, hue='species', ax=ax, kde=True)
#     ...

# ¿Alguna variable parece bimodal? ¿Por qué?

### Ejercicio 3 — Simulación de datos con diferente sesgo

Crea un DataFrame sintético con tres columnas:
- `simetrica`: 500 valores de una normal N(50, 10)
- `sesgo_positivo`: 500 valores de una log-normal (usa `np.random.lognormal`)
- `sesgo_negativo`: 500 valores de `100 - np.random.lognormal(...)`

Calcula el sesgo de cada columna con `.skew()` y visualiza los tres histogramas. ¿La función `diagnostico_forma()` detecta correctamente el sesgo?

In [ ]:
# np.random.seed(0)
# df_sintetico = pd.DataFrame({
#     'simetrica': np.random.normal(50, 10, 500),
#     'sesgo_positivo': np.random.lognormal(3, 0.8, 500),
#     'sesgo_negativo': 100 - np.random.lognormal(1.5, 0.5, 500)
# })
# diagnostico_forma(df_sintetico, df_sintetico.columns.tolist())
# ...


---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| **Distribución normal** | Campana simétrica. Parámetros: μ y σ. Regla 68-95-99.7. |
| **Distribución uniforme** | Todos los valores equiprobables. Alerta si aparece en variable continua real. |
| **Distribución binomial** | Éxitos en n ensayos con probabilidad p. Converge a la normal con n grande. |
| **Sesgo positivo** | Cola derecha (valores altos poco frecuentes). Media > mediana. |
| **Sesgo negativo** | Cola izquierda (valores bajos poco frecuentes). Media < mediana. |
| **Leptocúrtica** | Colas pesadas: más outliers de lo esperado. Curtosis exceso > 0. |
| **Platicúrtica** | Colas ligeras: menos outliers. Curtosis exceso < 0. |
| **Umbral práctico** | \|Sesgo\| > 1 → transformación recomendada antes de entrenar. |

### 🔜 Próxima sesión
**Bloque 2 — Sesión 2:** Test de Shapiro-Wilk, gráfico Q-Q y transformaciones para normalizar datos (logaritmo, raíz cuadrada, Box-Cox).